# 12 · The masked network: where its training comes from

Notebook 10 trains the masked network - the tile search's N+1 plus a correction through a gate
opened where the search's motion leaves the change unexplained - on moving tiles of every kind
in the training split. Drawn scenes and recordings teach it different things: the drawn ones
rigid motion, stops and drags with nothing else changing; the recordings what a screen really
does, most of it not motion. This notebook trains the same network on each alone, from scratch,
on notebook 10's own tiles, and scores all three on every validation tile by kind.

- **drawn only**: the training tiles of kinds `page`, `desktop`, `photo`;
- **recorded only**: the training tiles of kinds `screen` and `video`;
- **both**: notebook 10's network.

Epochs are chosen as in notebook 10: on its validation tiles of every kind (moving, capped at
50 dB).

Needs: notebook 10's tiles (`tiles_train`, `tiles_val` in the data folder) and its checkpoint.
Trainings in `checkpoints/masked_real/`; nothing is exported.

In [1]:
"""Settings."""

import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from helpers import masked, models, recursive, runs, training
from helpers.recursive import RecursiveSearch
from helpers.training import DEVICE

STILL, SHARPNESS, CAP = 0.1, 10.0, 50.0
LEARNING_RATE, BATCH_SIZE, PATIENCE = 1e-3, 16, 4
CACHE = training.CHECKPOINT_DIR / "masked_real"
SEED = 0
KINDS = ("page", "desktop", "photo", "screen", "video")
RECORDED = ("screen", "video")
training.seed_everything(SEED)
manifest = training.load_manifest()
FRACTION = 1 / runs.received_every(manifest)


def tile_search():
    return models.TileSearch(4, 2, SHARPNESS, 2, FRACTION, still=STILL)


published = tile_search().to(DEVICE).eval()
pyramid = models.PyramidSearch(((4, 4, 1), (2, 1, 2), (1, 2, 4)), SHARPNESS, FRACTION, STILL, "equiangular",
                               whole=True).to(DEVICE).eval()
tiles = {split: masked.load(split) for split in ("train", "val")}
for split, (frames, meta) in tiles.items():
    kinds = np.array([manifest["splits"][split]["sources"][s]["kind"] for s in meta["source"]])
    meta["kind"] = kinds
    print(f"{split}: {len(frames)} moving tiles - " + ", ".join(f"{k} {(kinds == k).sum()}" for k in KINDS))

train: 9856 moving tiles - page 1440, desktop 1302, photo 928, screen 4158, video 2028
val: 1788 moving tiles - page 194, desktop 96, photo 120, screen 1114, video 264


## 1. Training

As notebook 10: Adam from 1e-3, batches of 16 whole tiles, `metrics.luma_chroma_l1` over the window
less the halo, an epoch kept if it beats the best so far on the validation tiles; from the search
(the correction at zero).

In [2]:
def masked_network(state=None):
    network = models.Masked(tile_search(), gated=True).to(DEVICE)
    if state is not None:
        network.load_state_dict(state)
    return network


frames, meta = tiles["train"]
recorded_tiles = np.isin(meta["kind"], RECORDED)
SUBSETS = {"drawn only": ~recorded_tiles, "recorded only": recorded_tiles}
CACHE.mkdir(parents=True, exist_ok=True)
networks, histories = {}, {}
for name, chosen in SUBSETS.items():
    print(f"\n{'=' * 90}\n{name}: {chosen.sum()} training tiles\n{'=' * 90}")
    training.seed_everything(SEED)
    networks[name] = masked_network()
    histories[name], _ = masked.train(networks[name], frames[chosen], tiles["val"],
                                      CACHE / f"{name.replace(' ', '_')}.pt", LEARNING_RATE, BATCH_SIZE, PATIENCE)
state = torch.load(training.CHECKPOINT_DIR / "masked_runs" / "masked_gated.pt", map_location="cpu", weights_only=False)
networks["both (notebook 10)"] = masked_network(state["progress"]["best"]["state"])
histories["both (notebook 10)"] = state["progress"]["history"]
for name, network in networks.items():
    a, b = network.gate.detach().cpu().tolist()
    print(f"{name:<22} gate g = {1 / (1 + np.exp(-b)):.2f} where the search explains everything, "
          f"{1 / (1 + np.exp(-(a + b))):.2f} where it explains nothing")


drawn only: 3670 training tiles
start (the search, or the weights it was given): 41.309 dB on the moving validation tiles, capped
epoch 1: loss 0.01003, 41.309 dB, lr 1.0e-03, gate a, b -0.03, -0.03, 7s
epoch 2: loss 0.00991, 41.309 dB, lr 1.0e-03, gate a, b -0.04, -0.04, 7s
epoch 3: loss 0.00993, 41.309 dB, lr 1.0e-03, gate a, b -0.06, -0.06, 7s
epoch 4: loss 0.00994, 41.309 dB, lr 1.0e-03, gate a, b -0.08, -0.08, 7s
epoch 5: loss 0.00992, 41.309 dB, lr 5.0e-04, gate a, b -0.08, -0.09, 7s
epoch 6: loss 0.00987, 41.309 dB, lr 5.0e-04, gate a, b -0.09, -0.10, 7s
epoch 7: loss 0.00985, 41.309 dB, lr 5.0e-04, gate a, b -0.10, -0.11, 7s
epoch 8: loss 0.00990, 41.309 dB, lr 5.0e-04, gate a, b -0.10, -0.12, 7s
epoch 9: loss 0.00990, 41.309 dB, lr 2.5e-04, gate a, b -0.11, -0.12, 7s
epoch 10: loss 0.00989, 41.309 dB, lr 2.5e-04, gate a, b -0.11, -0.12, 7s
epoch 11: loss 0.00994, 41.309 dB, lr 2.5e-04, gate a, b -0.11, -0.13, 7s
epoch 12: loss 0.00985, 41.309 dB, lr 2.5e-04, gate a, b -0.11, 

## 2. Validation by kind

Every client tile of the validation clips (`recursive.load`), every method on the same frames:
repeat, the published search, and the three networks. Absolute wPSNR on the changed tiles and on
the moving ones capped at 50 dB, per kind; and each network's gain over the search on the moving
tiles with its 95% range over videos (clips for the drawn kinds).

In [3]:
def as_picture(model):
    model = model.to(DEVICE).eval()
    return lambda previous, current, target: model(previous, current)


PICTURES = {"repeat": lambda previous, current, target: current,
            "published search": as_picture(models.Motion(search=published, learned=False)),
            **{f"masked, {name}": as_picture(network) for name, network in networks.items()}}
path = CACHE / "val.pkl"
if path.exists() and path.stat().st_mtime > max((CACHE / f"{n.replace(' ', '_')}.pt").stat().st_mtime for n in SUBSETS):
    val = pickle.loads(path.read_bytes())
else:
    started = time.time()
    val = recursive.load(manifest, "val", {}, PICTURES, published, pyramid, STILL)
    path.write_bytes(pickle.dumps(val))
    print(f"{len(val['tiles']['source'])} tiles in {time.time() - started:.0f}s")
sources = manifest["splits"]["val"]["sources"]
t, w = val["tiles"], val["wpsnr"]
kind = np.array([sources[s]["kind"] for s in t["source"]])
group = np.array([sources[s].get("video", sources[s]["file"]) for s in t["source"]])
for label, mask, cap in (("changed tiles", t["changed"], False), ("moving tiles, ≤ 50 dB", t["moving"], True)):
    print(f"\n{label}")
    print(f"{'':<34}" + "".join(f"{k:>10}" for k in (*KINDS, "all")))
    print(f"{'tiles':<34}" + "".join(f"{(mask & (kind == k)).sum():10}" for k in KINDS) + f"{mask.sum():10}")
    for name in PICTURES:
        s = np.minimum(w[name], CAP) if cap else w[name]
        print(f"{name:<34}" + "".join(f"{s[mask & (kind == k)].mean():10.2f}" for k in KINDS) + f"{s[mask].mean():10.2f}")

rng = np.random.default_rng(SEED)


def over(gain, mask, draws=2000):
    per = [gain[mask & (group == g)] for g in np.unique(group[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


print("\nover the published search, moving tiles (≤ 50 dB), 95% over videos")
for name in [n for n in PICTURES if n.startswith("masked")]:
    gain = np.minimum(w[name], CAP) - np.minimum(w["published search"], CAP)
    parts = []
    for label, kinds in (("drawn", ("page", "desktop", "photo")), ("recorded", RECORDED)):
        m = t["moving"] & np.isin(kind, kinds)
        low, high = over(gain, m)
        parts.append(f"{label} {gain[m].mean():+.2f} [{low:+.2f}, {high:+.2f}]")
    print(f"  {name:<32}" + "   ".join(parts))

17190 tiles in 218s

changed tiles
                                        page   desktop     photo    screen     video       all
tiles                                   1310       184       960      1657      1458      5569
repeat                                 24.70     29.30     23.58     32.39     30.19     28.38
published search                       29.34     29.33     31.07     34.17     30.90     31.49
masked, drawn only                     29.34     29.33     31.07     34.17     30.90     31.49
masked, recorded only                  28.37     29.46     30.09     33.94     31.19     31.10
masked, both (notebook 10)             28.99     29.58     30.75     34.16     31.27     31.45

moving tiles, ≤ 50 dB
                                        page   desktop     photo    screen     video       all
tiles                                   1682       265      1297      4017      1835      9096
repeat                                 31.34     37.00     30.56     42.07     33.99   

## What it says

- **Trained on the drawn tiles alone, the network learns nothing.** No epoch beats the search on the
  validation tiles (its gate stays at ½, its correction at zero): what the drawn scenes teach - rigid
  motion, stops, drags - the search already does, and their unsure cells hold nothing a correction
  could fix.
- **Trained on the recorded tiles alone, it learns the recordings and spoils the drawn scenes.** +0.07
  dB over the search on the recorded moving tiles (95% over videos −0.06 to +0.20), −0.36 on the drawn
  ones (−0.60 to −0.16): its correction, drawn for a screen's fades and redraws, blurs a scroll. Its
  gate stays open almost everywhere (0.39 where the search explains the change, 0.56 where it does
  not).
- **Trained on both (notebook 10), it is the least bad of the three**: +0.12 dB on the recorded moving
  tiles (−0.01 to +0.25), −0.03 on the drawn ones, +0.07 over all; video +0.29, screens +0.05. Both
  kinds of tile teach it where to stay quiet - its gate closes to 0.13 where the search explains the
  change.

So where the training comes from decides where the network does harm, not whether it helps: on every
source it is within a tenth of a dB of the search, for fourteen times the work.